# Synthetic Model — Finite-Difference Modeling

A simple 2D model, built on purpose, to test robustness and (later)
lateral continuity: a flat water bottom, gently dipping interfaces and
a normal fault. The data are generated by **acoustic finite-difference
modeling** (Devito): the wavefield is propagated through the model, so
primaries, free-surface multiples, diffractions and head waves appear
on their own — unlike the 1D generator, which draws one hyperbola per
reflector.

**How the data were made** (`scripts/model_fd.py`, config
`configs/synthetic_fd_model.json`):
1. one shot per offset, source at `cdp_x - h/2`, receivers along the
   whole surface;
2. the trace at `cdp_x + h/2` of each shot goes into the CDP gather (all
   50 traces share the midpoint `cdp_x`);
3. direct-wave mute and a time gain;
4. the CDP gather is saved as a test case in `cases/` — the algorithms
   run on it unchanged — with the semblance peak of each expected event
   as the evaluation reference.

Two versions of every gather: **with free surface** (surface multiples)
and **without** (absorbing top). Comparing them shows which events are
multiples.

**Execution order:**
1. Setup
2. Parameters
3. The model
4. Expected events
5. Load the modeled data
6. Shot records (seismograms)
7. CDP gathers
8. Traces (wiggle display)
9. Semblance panels
10. Evaluation reference: the semblance peaks
11. Side by side: shot, CDP gather and semblance
12. Summary


## 1. Setup

In [ ]:
import sys
import os
import json
import numpy as np
import matplotlib.pyplot as plt

# Project root: walk up to the folder that contains src/
from pathlib import Path
project_root = Path.cwd().resolve()
while not (project_root / 'src').is_dir():
    if project_root.parent == project_root:
        raise RuntimeError('Project root (folder containing src/) not found.')
    project_root = project_root.parent
project_root = str(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f'Project root: {project_root}')

In [ ]:
from src.seismic.model2d   import build_layered_model, column_vrms, column_event_times, surface_multiples
from src.seismic.cases     import load_case
from src.seismic.semblance import compute_semblance

print('All modules imported successfully.')

## 2. Parameters

In [ ]:
CONFIG   = os.path.join(project_root, 'configs', 'synthetic_fd_model.json')
CDP_X    = 2500.0        # CDP position (m); must be one of acquisition.cdp_x_m

VEL_MIN  = 1400.0        # semblance scan (m/s)
VEL_MAX  = 3000.0
VEL_STEP = 20.0

with open(CONFIG) as f:
    cfg = json.load(f)
acq = cfg['acquisition']

CASES = {variant: f"{cfg['name']}_cdp{int(CDP_X):04d}_{variant}" for variant in ('fs', 'nofs')}
LABEL = {'fs': 'with free surface', 'nofs': 'without free surface'}

print('Model     :', cfg['description'])
print('Cases     :', CASES)

## 3. The model

Velocity section with the interfaces, the CDP position and the span of
the sources (left of the CDP) and receivers (right of the CDP) that
build its gather. The column under the CDP gives the ground-truth
V_rms.

In [ ]:
model = build_layered_model(cfg['model'])
x, z, vp = model['x'], model['z'], model['vp']

offsets = np.linspace(acq['offset_min_m'], acq['offset_max_m'], acq['n_offsets'])
src_span = (CDP_X - offsets.max() / 2, CDP_X - offsets.min() / 2)
rec_span = (CDP_X + offsets.min() / 2, CDP_X + offsets.max() / 2)

t_col, v_rms_col = column_vrms(model, CDP_X, acq['dt_ms'], acq['t_max_ms'])
profile = vp[int(round(CDP_X / model['dx']))]

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), gridspec_kw={'width_ratios': [4, 1]})

im = axes[0].imshow(vp.T, aspect='auto', cmap='viridis',
                    extent=[x[0], x[-1], z[-1], z[0]])
for name, d in zip(model['names'], model['interfaces']):
    axes[0].plot(x, d, 'w-', linewidth=0.8)
axes[0].axvline(CDP_X, color='red', linewidth=1.2, label=f'CDP at {CDP_X:.0f} m')
axes[0].plot(src_span, [15, 15], color='orange', linewidth=4, label='sources of this CDP')
axes[0].plot(rec_span, [15, 15], color='cyan', linewidth=4, label='receivers of this CDP')
axes[0].set_xlabel('Distance (m)')
axes[0].set_ylabel('Depth (m)')
axes[0].set_title('P-wave velocity model')
axes[0].legend(loc='lower left', fontsize=8)
fig.colorbar(im, ax=axes[0], fraction=0.03, pad=0.01, label='Velocity (m/s)')

axes[1].plot(profile, z, color='black', linewidth=1.2, label='Interval velocity')
axes[1].set_ylim(z[-1], z[0])
axes[1].set_xlabel('Velocity (m/s)')
axes[1].set_ylabel('Depth (m)')
axes[1].set_title(f'Column at {CDP_X:.0f} m')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 4. Expected events

Primaries (zero-offset time and V_rms of each interface under the CDP)
and the main free-surface multiples: water-bottom multiples (only in
the water, at the water velocity) and first-order peg-legs (a primary
plus one extra bounce in the water). These are drawn on the semblance
panels below, as `o` and `x`.

In [ ]:
primaries = column_event_times(model, CDP_X)
multiples = surface_multiples(primaries)

print(f'{"Primary":>14} {"depth (m)":>10} {"t0 (ms)":>9} {"V_rms (m/s)":>12}')
for p in primaries:
    print(f"{p['name']:>14} {p['depth_m']:>10.0f} {p['t0_ms']:>9.0f} {p['v_rms']:>12.0f}")
print()
print(f'{"Multiple":>28} {"t0 (ms)":>9} {"V_nmo (m/s)":>12}')
for m in multiples:
    print(f"{m['name']:>28} {m['t0_ms']:>9.0f} {m['v_nmo']:>12.0f}")


## 5. Load the modeled data

The CDP gathers are test cases in `cases/`; the shot records (three
shots per gather: near, middle and far offset) are in
`data/synthetic_fd/`. If they are missing, generate them with
`python scripts/model_fd.py` (needs Devito; a few minutes).

In [ ]:
data = {}
for variant, name in CASES.items():
    case_path = os.path.join(project_root, 'cases', f'{name}.npz')
    shot_path = os.path.join(project_root, 'data', 'synthetic_fd', f'{name}_shots.npz')
    if not (os.path.exists(case_path) and os.path.exists(shot_path)):
        raise FileNotFoundError(f'{name} not found: run  python scripts/model_fd.py  first.')
    with np.load(shot_path) as s:
        data[variant] = {'case': load_case(case_path),
                         **{k: s[k] for k in s.files}}

t_grid = data['fs']['case']['t_grid']
print({v: d['case']['gather'].shape for v, d in data.items()})
print('Shots kept for display (source x, m):', data['fs']['shot_x'])


## 6. Shot records (seismograms)

One shot, all receivers of the regular line. Without the free surface
the **direct wave** (straight lines from the source) dominates; with
it, the source and receivers sit just below a surface that reflects
everything back down, which weakens the direct wave and creates the
**surface multiples** (repeated copies of the reflections). A time
gain is applied for display only.

In [ ]:
SHOT = 1     # 0 = near-offset shot, 1 = middle, 2 = far-offset shot

fig, axes = plt.subplots(1, 2, figsize=(15, 7), sharey=True)
for ax, variant in zip(axes, ('fs', 'nofs')):
    d   = data[variant]
    li  = d['line_idx']
    rx  = d['rec_x'][li]
    rec = d['shots'][SHOT][li] * (d['t_ms'] / d['t_ms'][-1])[None, :]
    clip = np.percentile(np.abs(rec), 99)
    ax.imshow(rec.T, aspect='auto', cmap='gray', vmin=-clip, vmax=clip,
              extent=[rx[0], rx[-1], t_grid[-1], t_grid[0]])
    ax.axvline(d['shot_x'][SHOT], color='red', linewidth=0.8, label='source')
    ax.set_xlabel('Receiver position (m)')
    ax.set_title(f"Shot at {d['shot_x'][SHOT]:.0f} m — {LABEL[variant]}")
    ax.legend(loc='lower left')
axes[0].set_ylabel('Time (ms)')
plt.tight_layout()
plt.savefig(os.path.join(project_root, 'results', 'figures', 'fd_shot_records.png'), dpi=150)
plt.show()

## 7. CDP gathers

Left: the traces exactly as taken from the shots (raw). Right: after the
direct-wave mute and the time gain — the gathers the algorithms use.

In [ ]:
os.makedirs(os.path.join(project_root, 'results', 'figures'), exist_ok=True)

fig, axes = plt.subplots(1, 4, figsize=(18, 7), sharey=True)
panels = [('fs', 'raw'), ('fs', 'processed'), ('nofs', 'raw'), ('nofs', 'processed')]
for ax, (variant, kind) in zip(axes, panels):
    d = data[variant]
    g = d['raw_cdp_gather'] if kind == 'raw' else d['case']['gather']
    off = d['case']['offsets']
    clip = np.percentile(np.abs(g), 99)
    ax.imshow(g.T, aspect='auto', cmap='gray', vmin=-clip, vmax=clip,
              extent=[off[0], off[-1], t_grid[-1], t_grid[0]])
    ax.set_xlabel('Offset (m)')
    ax.set_title(f'{LABEL[variant]}\n{kind}', fontsize=10)
axes[0].set_ylabel('Time (ms)')
plt.suptitle(f'CDP gathers at {CDP_X:.0f} m', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(project_root, 'results', 'figures', 'fd_cdp_gathers.png'), dpi=150)
plt.show()

## 8. Traces (wiggle display)

The same gather as individual traces: each curve is the signal recorded
by one receiver, plotted along time. A reflection is a wavelet that
appears on every trace, later on the far ones (moveout). Every 4th
trace is shown.

In [ ]:
EVERY = 4

fig, axes = plt.subplots(1, 2, figsize=(15, 8), sharey=True)
for ax, variant in zip(axes, ('fs', 'nofs')):
    c   = data[variant]['case']
    g   = c['gather'][::EVERY]
    off = c['offsets'][::EVERY]
    scale = 0.9 * np.diff(off).mean() / np.percentile(np.abs(g), 99.5)
    spacing = np.diff(off).mean()
    for h, tr in zip(off, g):
        w = h + np.clip(tr * scale, -1.2 * spacing, 1.2 * spacing)   # no overlap
        ax.plot(w, t_grid, color='black', linewidth=0.5)
        ax.fill_betweenx(t_grid, h, w, where=w > h, color='black', linewidth=0)
    ax.set_ylim(t_grid[-1], t_grid[0])
    ax.set_xlabel('Offset (m)')
    ax.set_title(f'CDP {CDP_X:.0f} m — {LABEL[variant]}')
axes[0].set_ylabel('Time (ms)')
plt.tight_layout()
plt.show()


## 9. Semblance panels

`o`: expected primaries; `x`: expected surface multiples; dashed line:
V_rms of the column under the CDP. Below the last interface (top C)
there are no more primaries: with the free surface, everything bright
there is a multiple — the trap the objective function has to resist.
Without the free surface, the remaining late energy comes from
diffractions at the fault (the receivers of this CDP cross it) and
weak interbed multiples.

In [ ]:
velocities = np.arange(VEL_MIN, VEL_MAX + VEL_STEP, VEL_STEP, dtype=np.float32)
for d in data.values():
    c = d['case']
    d['panel'] = compute_semblance(c['gather'], c['offsets'], velocities, c['dt_ms'])

fig, axes = plt.subplots(1, 2, figsize=(14, 8), sharey=True)
for ax, variant in zip(axes, ('fs', 'nofs')):
    c = data[variant]['case']
    ax.imshow(data[variant]['panel'].T, aspect='auto', cmap='jet', vmin=0, vmax=1,
              extent=[velocities[0], velocities[-1], t_grid[-1], t_grid[0]])
    ax.plot(c['v_rms_true'], t_grid, 'w--', linewidth=1.2, label='V_rms (column)')
    ax.plot([p['v_rms'] for p in primaries], [p['t0_ms'] for p in primaries],
            'o', mfc='none', mec='white', markersize=11, mew=1.5, label='primaries')
    ax.plot([m['v_nmo'] for m in multiples], [m['t0_ms'] for m in multiples],
            'x', color='magenta', markersize=10, mew=2, label='surface multiples')
    ax.set_xlabel('Velocity (m/s)')
    ax.set_title(f'Semblance — {LABEL[variant]}')
axes[0].set_ylabel('Time (ms)')
axes[0].legend(loc='lower left', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(project_root, 'results', 'figures', 'fd_semblance.png'), dpi=150)
plt.show()


## 10. Evaluation reference: the semblance peaks

The `o` marks above are where the model says each event is: its vertical
time t0 and the column V_rms. The semblance maxima ("blobs") are not
exactly there: the real moveout is not a perfect hyperbola (long
offsets, post-critical reflections), so the best-fitting hyperbola is
1–3% faster and slightly shifted in time. That best hyperbola — the
**stacking velocity** — is what a picker, human or algorithm, chooses.

So the reference used to evaluate the algorithms is the **semblance peak
of each expected event**: the highest local maximum inside a box of
±40 ms and ±8% around the model position (`src/analysis/reference.py`).
A local maximum, and not simply the largest value of the box, avoids
taking the flank of a stronger event that enters the box. The peaks are
stored in each case's meta (`meta['reference_picks']`, written by
`scripts/add_reference_picks.py`); the model only says where to look,
the algorithms never see them.

`o`: model position; `+`: reference (semblance peak). Primaries in white,
multiples in magenta.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 8), sharey=True)
for ax, variant in zip(axes, ('fs', 'nofs')):
    c    = data[variant]['case']
    refs = c['meta']['reference_picks']
    ax.imshow(data[variant]['panel'].T, aspect='auto', cmap='jet', vmin=0, vmax=1,
              extent=[velocities[0], velocities[-1], t_grid[-1], t_grid[0]])
    for r in refs:
        color = 'white' if r['kind'] == 'primary' else 'magenta'
        ax.plot(r['v_model'], r['t0_ms'], 'o', mfc='none', mec=color, markersize=11, mew=1.2)
        ax.plot(r['v'], r['t_ms'], '+', color=color, markersize=14, mew=2.2)
        ax.plot([r['v_model'], r['v']], [r['t0_ms'], r['t_ms']], '-', color=color, linewidth=0.8)
    ax.set_xlim(VEL_MIN, 2400)
    ax.set_xlabel('Velocity (m/s)')
    ax.set_title(f'Model position (o) vs semblance peak (+) — {LABEL[variant]}')
axes[0].set_ylabel('Time (ms)')
plt.tight_layout()
plt.savefig(os.path.join(project_root, 'results', 'figures', 'fd_reference_picks.png'), dpi=150)
plt.show()

for variant in ('fs', 'nofs'):
    print(f'--- {LABEL[variant]} ---')
    print(f'{"Event":>28} {"model t0":>9} {"model v":>8} {"peak t":>7} {"peak v":>7} '
          f'{"dv":>6} {"S":>5}')
    for r in data[variant]['case']['meta']['reference_picks']:
        flag = '  edge!' if r['on_edge'] else ''
        print(f"{r['name']:>28} {r['t0_ms']:>9.0f} {r['v_model']:>8.0f} {r['t_ms']:>7.0f} "
              f"{r['v']:>7.0f} {100 * (r['v'] / r['v_model'] - 1):>+5.1f}% {r['s']:>5.2f}{flag}")
    print()

## 11. Side by side: shot, CDP gather and semblance

The same events followed through the three views, for each version of
the data.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 13), sharey=True,
                         gridspec_kw={'width_ratios': [1.6, 1, 1]})
for row, variant in enumerate(('fs', 'nofs')):
    d, c = data[variant], data[variant]['case']
    li, rx = d['line_idx'], d['rec_x'][d['line_idx']]
    rec = d['shots'][SHOT][li] * (d['t_ms'] / d['t_ms'][-1])[None, :]
    clip = np.percentile(np.abs(rec), 99)
    axes[row, 0].imshow(rec.T, aspect='auto', cmap='gray', vmin=-clip, vmax=clip,
                        extent=[rx[0], rx[-1], t_grid[-1], t_grid[0]])
    axes[row, 0].axvline(d['shot_x'][SHOT], color='red', linewidth=0.8)
    axes[row, 0].set_title(f"Shot at {d['shot_x'][SHOT]:.0f} m — {LABEL[variant]}")
    axes[row, 0].set_ylabel('Time (ms)')

    g = c['gather']; clip = np.percentile(np.abs(g), 99)
    axes[row, 1].imshow(g.T, aspect='auto', cmap='gray', vmin=-clip, vmax=clip,
                        extent=[c['offsets'][0], c['offsets'][-1], t_grid[-1], t_grid[0]])
    axes[row, 1].set_title(f'CDP gather at {CDP_X:.0f} m')

    axes[row, 2].imshow(d['panel'].T, aspect='auto', cmap='jet', vmin=0, vmax=1,
                        extent=[velocities[0], velocities[-1], t_grid[-1], t_grid[0]])
    axes[row, 2].plot(c['v_rms_true'], t_grid, 'w--', linewidth=1.2)
    axes[row, 2].plot([p['v_rms'] for p in primaries], [p['t0_ms'] for p in primaries],
                      'o', mfc='none', mec='white', markersize=11, mew=1.5)
    if variant == 'fs':
        axes[row, 2].plot([m['v_nmo'] for m in multiples], [m['t0_ms'] for m in multiples],
                          'x', color='magenta', markersize=10, mew=2)
    axes[row, 2].set_title('Semblance')
for ax, label in zip(axes[1], ('Receiver position (m)', 'Offset (m)', 'Velocity (m/s)')):
    ax.set_xlabel(label)
plt.tight_layout()
plt.savefig(os.path.join(project_root, 'results', 'figures', 'fd_side_by_side.png'), dpi=150)
plt.show()
print('Figures saved in results/figures/.')


## 12. Summary

**What the modeled data contain.**
- The four primaries arrive at the times predicted by the model
  (`o` on the panels).
- With the free surface, the water-bottom multiple and the peg-legs show
  up as semblance maxima at velocities lower than the true V_rms — and
  below the last interface they are the only coherent energy left.
- Without the free surface those maxima disappear; the late energy that
  remains comes from diffractions at the fault and weak interbed
  multiples.

**Ground truth.** `v_rms_true` is the V_rms of the vertical column under
the CDP, but the semblance peaks sit 1–3% higher and up to ~35 ms off in
time: the moveout of the modeled events is not a perfect hyperbola (long
offsets, post-critical reflections). Dip adds little (V / cos(dip) is
below half a percent for 3–5°). The algorithms maximize semblance, so
they are evaluated against the **semblance peak of each expected event**
(`meta['reference_picks']`, `src.analysis.reference.match_reference`):
the stacking velocity a picker would choose, not the model V_rms.

**Modeling notes.**
- The absorbing boundaries still reflect a small part of the energy
  (~0.5% in a homogeneous test), so the model is deeper than the record
  needs: the reflection from the bottom of the grid arrives after the
  end of the record.
- No random noise is added yet: the next ingredients (noise types,
  more CDPs along the line) are switched on one at a time.

**Next steps.**
- More CDPs along the line, including the fault zone, for lateral
  continuity.
- Noise as separate, switchable ingredients.
- Run the v3 algorithms on `fd_layered_cdp2500_fs` vs `_nofs` to measure
  how much the multiples hurt.